In [ ]:
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import make_pipeline
from sklearn.linear_model import LogisticRegression
from sklearn.naive_bayes import GaussianNB
from sklearn.metrics import accuracy_score

# Load dataset (no header in file)
cols = [
    "sepal_length",
    "sepal_width",
    "petal_length",
    "petal_width",
    "species",
]

df = pd.read_csv("DSBDAL_datasets/iris.csv", header=None, names=cols)

# e) Data cleaning: remove NA, ?, negative values

df = df.replace("?", np.nan)
for col in cols[:-1]:
    df[col] = pd.to_numeric(df[col], errors="coerce")
    df.loc[df[col] < 0, col] = np.nan

df = df.dropna()

# f) Error correcting: simple outlier removal (IQR)

def iqr_filter(data, col):
    q1 = data[col].quantile(0.25)
    q3 = data[col].quantile(0.75)
    iqr = q3 - q1
    low = q1 - 1.5 * iqr
    high = q3 + 1.5 * iqr
    return data[(data[col] >= low) & (data[col] <= high)]

for c in cols[:-1]:
    df = iqr_filter(df, c)

# g) Data transformation: scale features

X = df.drop(columns=["species"])
y = df["species"]

# h) Models: Logistic Regression and Naive Bayes

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

log_reg = make_pipeline(StandardScaler(), LogisticRegression(max_iter=1000))
nb = GaussianNB()

log_reg.fit(X_train, y_train)
nb.fit(X_train, y_train)

pred_lr = log_reg.predict(X_test)
pred_nb = nb.predict(X_test)

print("Logistic Regression accuracy:", accuracy_score(y_test, pred_lr))
print("Naive Bayes accuracy:", accuracy_score(y_test, pred_nb))